In [6]:
# saving the raw_pay_gap_df in processed folder (Git dekstop)

import pandas as pd
from pathlib import Path

pay_gap_input_df = pd.read_csv(r"C:\Users\madha\Desktop\Projects\UK_Gender_PayGap_BonusOutcomes_and_Representation Analytics\processed\pay_gap_step1.csv")
pay_gap_input_df.head()

,employee_id,gender,department,job_title,job_level,years_at_company,contracted_hours_per_week,work_location,ethnicity,annual_base_pay_gbp,annual_bonus_gbp,total_annual_pay_gbp,employment_type,hire_date
0,E000001,Female,Data,Analyst,4,15.0,25.0,Birmingham,Other,63500,7121.05,70621.05,Part-time,2018-03-18
1,E000002,Male,HR,Analyst,1,13.3,20.0,Bristol,Black,26800,1924.97,28724.97,Fixed-term,2019-07-20
2,E000003,Male,Support,Analyst,3,8.9,40.0,London,White,35200,5334.13,40534.13,Fixed-term,2023-02-18
3,E000004,Female,Operations,Manager,4,2.9,40.0,London,Mixed,51500,2349.34,53849.34,Fixed-term,2019-12-14
4,E000005,Female,Product,Lead,1,2.3,40.0,Leeds,Prefer not to say,35200,2306.93,37506.93,Permanent,2026-02-07


In [25]:
# Checking the date other than hire date 

hire_date_text = pay_gap_input_df['hire_date'].astype('string').str.strip()
correct_hire_date = pay_gap_input_df['hire_date'].astype('string').str.strip().str.fullmatch(r'[0-9]{4}-[0-9]{2}-[0-9]{2}',na=False)
incorrect_hire_date_df = pay_gap_input_df.loc[~correct_hire_date,['employee_id','hire_date']]
display(incorrect_hire_date_df)

,employee_id,hire_date


In [26]:
# Unparsed hire date 

parsed_hire_date = pd.to_datetime(hire_date_text, format='%Y-%m-%d', errors='coerce')

invalid_hire_date = ~correct_hire_date | parsed_hire_date.isna()
unparsed_hire_date = parsed_hire_date.isna()
print(f'Total invalid date:',invalid_hire_date.sum())
print(f'Total unparsed date :', (unparsed_hire_date.sum()))

Total invalid date: 0
Total unparsed date : 0


In [27]:
# checking the employee id to put in rejected csv

employee_id = pay_gap_input_df['employee_id'].astype('string').str.strip()

null_employee_id = employee_id.isnull()
blank_employee_id = employee_id.eq('').fillna(False)

missing_employee_id = null_employee_id |blank_employee_id
print("Rows with null employee IDs:", null_employee_id.sum())
print("Rows with blank employee IDs:", blank_employee_id.sum())
print("Total rows with missing employee IDs:", missing_employee_id.sum())

Rows with null employee IDs: 0
Rows with blank employee IDs: 0
Total rows with missing employee IDs: 0


In [28]:
duplicate_employee_id = ~missing_employee_id & employee_id.duplicated(keep=False)
print("Rows with duplicate employee IDs:", duplicate_employee_id.sum())

Rows with duplicate employee IDs: 0


In [29]:
pay_gap_input_df['job_level'].dtype

dtype('int64')

In [31]:
invalid_job_level = ~(pay_gap_input_df['job_level'].isin([1,2,3,4,5]))
print("Rows with invalid job levels:", invalid_job_level.sum())

Rows with invalid job levels: 0


In [32]:
contracted_hours = pay_gap_input_df['contracted_hours_per_week']

zero_hours = contracted_hours.eq(0)
negative_hours = contracted_hours.lt(0)
above_ceiling_hours = contracted_hours.gt(80)

invalid_contracted_hours = zero_hours | negative_hours | above_ceiling_hours

print("Rows with zero hours:", zero_hours.sum())
print("Rows with negative hours:", negative_hours.sum())
print("Rows above 80 hours:", above_ceiling_hours.sum())
print("Total rows failing the hours check:", invalid_contracted_hours.sum())

Rows with zero hours: 0
Rows with negative hours: 0
Rows above 80 hours: 0
Total rows failing the hours check: 0


In [34]:
annual_base_pay = pay_gap_input_df['annual_base_pay_gbp']

zero_base_pay = annual_base_pay.eq(0)
negative_base_pay = annual_base_pay.lt(0)

invalid_base_pay = zero_base_pay |negative_base_pay
print('Invalid base pay is :',invalid_base_pay.sum())

Invalid base pay is : 120


In [35]:
annual_bonus_pay = pay_gap_input_df['annual_bonus_gbp']

invalid_bonus_pay = annual_bonus_pay.lt(0)
print('Invalid bonus pay is :',invalid_bonus_pay.sum())

Invalid bonus pay is : 0


In [38]:
pay_columns = [
    "annual_base_pay_gbp",
    "annual_bonus_gbp",
    "total_annual_pay_gbp",
]

original_pay_df = pay_gap_input_df[pay_columns]
numeric_pay_df = original_pay_df.apply(pd.to_numeric, errors="coerce")

non_numeric_pay_df = original_pay_df.notna() & numeric_pay_df.isna()

print("Non-numeric values by pay column:")
display(non_numeric_pay_df.sum())

invalid_non_numeric_pay = non_numeric_pay_df.any(axis=1)
print("Rows with any non-numeric pay value:", invalid_non_numeric_pay.sum())


Non-numeric values by pay column:


annual_base_pay_gbp     0
annual_bonus_gbp        0
total_annual_pay_gbp    0
dtype: int64

Rows with any non-numeric pay value: 0


In [40]:
expected_total_gbp = (
    pay_gap_input_df["annual_base_pay_gbp"]
    + pay_gap_input_df["annual_bonus_gbp"]
)

difference_pay = (
    pay_gap_input_df["total_annual_pay_gbp"] - expected_total_gbp
).abs().round(2)

invalid_total_pay = difference_pay.gt(0.01)

print("Rows where total pay differs by more than £0.01:", invalid_total_pay.sum())

Rows where total pay differs by more than £0.01: 120


In [41]:
failed_both = invalid_base_pay & invalid_total_pay
failed_either = invalid_base_pay | invalid_total_pay

print("Rows failing both pay rules:", failed_both.sum())
print("Unique rows failing either pay rule:", failed_either.sum())

Rows failing both pay rules: 120
Unique rows failing either pay rule: 120


In [44]:
pay_gap_input_df['annual_base_pay_gbp'].eq(0).sum()
pay_gap_input_df['annual_base_pay_gbp'].lt(0).sum()

np.int64(120)

In [49]:
negative_base_pay = pay_gap_input_df['annual_base_pay_gbp'].lt(0)
reject_ = pay_gap_input_df[negative_base_pay]
reject

employee_id                  E000154E000415E002691E002963E003557E003719E004...
gender                       MaleFemaleMaleFemaleFemaleMaleMaleFemaleFemale...
department                   LegalProductProductOperationsFinanceOperations...
job_title                    SpecialistManagerSpecialistAnalystManagerLeadM...
job_level                                                                  282
years_at_company                                                        1444.9
contracted_hours_per_week                                               3650.0
work_location                ManchesterBirminghamManchesterGlasgowLondonLee...
ethnicity                    AsianMixedWhiteMixedOtherPrefer not to sayBlac...
annual_base_pay_gbp                                                     -12000
annual_bonus_gbp                                                      420372.9
total_annual_pay_gbp                                                 5307672.9
employment_type              Fixed-termFixed-termPar

In [50]:
# Add the source row number if you haven't already added it.
if "source_row_number" not in pay_gap_input_df.columns:
    pay_gap_input_df["source_row_number"] = range(2, len(pay_gap_input_df) + 2)

hard_failure_checks_df = pd.DataFrame({
    "INVALID_HIRE_DATE": invalid_hire_date,
    "MISSING_EMPLOYEE_ID": missing_employee_id,
    "DUPLICATE_EMPLOYEE_ID": duplicate_employee_id,
    "INVALID_JOB_LEVEL": invalid_job_level,
    "INVALID_CONTRACTED_HOURS": invalid_contracted_hours,
    "BASE_PAY_ZERO_OR_NEGATIVE": invalid_base_pay,
    "BONUS_PAY_NEGATIVE": invalid_bonus_pay,
    "NON_NUMERIC_PAY": invalid_non_numeric_pay,
    "TOTAL_PAY_MISMATCH": invalid_total_pay,
})

rejected_mask = hard_failure_checks_df.any(axis=1)
rejected_pay_gap_df = pay_gap_input_df.loc[rejected_mask].copy()

rejected_pay_gap_df["rejection_reason_codes"] = (
    hard_failure_checks_df.loc[rejected_mask]
    .apply(lambda row: " | ".join(row.index[row]), axis=1)
)

rejected_pay_gap_df.to_csv(
    r"C:\Users\madha\Desktop\Projects\UK_Gender_PayGap_BonusOutcomes_and_Representation Analytics\rejected\rejected_pay_gap.csv",
    index=False,
)

print("Rows exported to rejected CSV:", len(rejected_pay_gap_df))
display(rejected_pay_gap_df[
    ["source_row_number", "employee_id", "rejection_reason_codes"]
].head())

Rows exported to rejected CSV: 120


,source_row_number,employee_id,rejection_reason_codes
153,155,E000154,BASE_PAY_ZERO_OR_NEGATIVE | TOTAL_PAY_MISMATCH
414,416,E000415,BASE_PAY_ZERO_OR_NEGATIVE | TOTAL_PAY_MISMATCH
2690,2692,E002691,BASE_PAY_ZERO_OR_NEGATIVE | TOTAL_PAY_MISMATCH
2962,2964,E002963,BASE_PAY_ZERO_OR_NEGATIVE | TOTAL_PAY_MISMATCH
3556,3558,E003557,BASE_PAY_ZERO_OR_NEGATIVE | TOTAL_PAY_MISMATCH


In [53]:
# creating review_pay_gap_warnings.csv  

accepted_pay_gap_df = pay_gap_input_df.loc[~rejected_mask].copy()
work_location = accepted_pay_gap_df['work_location'].astype('string').str.strip()
missing_work_location = work_location.isna() | work_location.eq('')

accepted_pay_gap_df['work_location_warning_flag'] = missing_work_location
accepted_pay_gap_df.loc[missing_work_location,'work_location']='Not Recorded'
print("Accepted employees with a work location warning:", missing_work_location.sum())

Accepted employees with a work location warning: 250


In [54]:
accepted_pay_gap_df.head()

,employee_id,gender,department,job_title,job_level,years_at_company,contracted_hours_per_week,work_location,ethnicity,annual_base_pay_gbp,annual_bonus_gbp,total_annual_pay_gbp,employment_type,hire_date,source_row_number,work_location_warning_flag
0,E000001,Female,Data,Analyst,4,15.0,25.0,Birmingham,Other,63500,7121.05,70621.05,Part-time,2018-03-18,2,False
1,E000002,Male,HR,Analyst,1,13.3,20.0,Bristol,Black,26800,1924.97,28724.97,Fixed-term,2019-07-20,3,False
2,E000003,Male,Support,Analyst,3,8.9,40.0,London,White,35200,5334.13,40534.13,Fixed-term,2023-02-18,4,False
3,E000004,Female,Operations,Manager,4,2.9,40.0,London,Mixed,51500,2349.34,53849.34,Fixed-term,2019-12-14,5,False
4,E000005,Female,Product,Lead,1,2.3,40.0,Leeds,Prefer not to say,35200,2306.93,37506.93,Permanent,2026-02-07,6,False


In [58]:
accepted_pay_gap_df['gender'].isna().sum()
accepted_pay_gap_df['gender'].unique().tolist()

['Female', 'Male', 'Prefer not to say', 'Non-binary', 'unknown']

In [62]:
gender = accepted_pay_gap_df['gender'].astype('string').str.strip().str.lower()

gender_warning = gender.eq('unknown')

accepted_pay_gap_df['gender_warning_flag']= gender_warning
accepted_pay_gap_df.loc[gender_warning,'gender']='Not Recorded'
print('Accepted rows with gender warning :',gender_warning.sum())


Accepted rows with gender warning : 160


In [63]:
# write review csv 

warning_rules_df = pd.DataFrame({
    "MISSING_WORK_LOCATION": accepted_pay_gap_df["work_location_warning_flag"],
    "MISSING_OR_INCONSISTENT_GENDER": accepted_pay_gap_df["gender_warning_flag"],})

review_mask = warning_rules_df.any(axis=1)
review_pay_gap_df = accepted_pay_gap_df.loc[review_mask].copy()

review_pay_gap_df["warning_reason_codes"] = (
    warning_rules_df.loc[review_mask]
    .apply(lambda row: " | ".join(row.index[row]), axis=1)
)

review_pay_gap_df.to_csv(
    r"C:\Users\madha\Desktop\Projects\UK_Gender_PayGap_BonusOutcomes_and_Representation Analytics\processed\review_pay_gap_warnings.csv",
    index=False,
)

print("Accepted rows with warnings:", len(review_pay_gap_df))
display(review_pay_gap_df[
    ["source_row_number", "employee_id", "warning_reason_codes"]
].head())

Accepted rows with warnings: 408


,source_row_number,employee_id,warning_reason_codes
58,60,E000059,MISSING_WORK_LOCATION
127,129,E000128,MISSING_WORK_LOCATION
224,226,E000225,MISSING_WORK_LOCATION
283,285,E000284,MISSING_OR_INCONSISTENT_GENDER
303,305,E000304,MISSING_WORK_LOCATION


In [64]:
input_count = len(pay_gap_input_df)
accepted_count = len(accepted_pay_gap_df)
rejected_count = len(rejected_pay_gap_df)
review_count = len(review_pay_gap_df)

print("Input rows:", input_count)
print("Accepted rows without warnings:", accepted_count - review_count)
print("Accepted rows with warnings:", review_count)
print("Total accepted rows:", accepted_count)
print("Rejected rows:", rejected_count)

assert accepted_count + rejected_count == input_count
assert review_pay_gap_df["source_row_number"].isin(
    accepted_pay_gap_df["source_row_number"]
).all()

print("Reconciliation passed.")

Input rows: 50000
Accepted rows without warnings: 49472
Accepted rows with warnings: 408
Total accepted rows: 49880
Rejected rows: 120
Reconciliation passed.


In [66]:
rejected_pay_gap_df.shape


(120, 16)

In [67]:
accepted_pay_gap_df.shape

(49880, 17)

In [69]:
review_pay_gap_df.shape

(408, 18)

In [68]:
pay_gap_input_df.shape

(50000, 15)

In [72]:
accepted_pay_gap_df.to_csv(
    r"C:\Users\madha\Desktop\Projects\UK_Gender_PayGap_BonusOutcomes_and_Representation Analytics\processed\accepted_pay_gap.csv",
    index=False,
)

print(f"Accepted rows saved: {len(accepted_pay_gap_df):,}")

Accepted rows saved: 49,880
